# Stage 3 — Cleaning & Preparing the Data

## Setup

The cell below imports the libraries and defines the file paths. 


In [ ]:
import os, re, time, gc
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.compute as pc

BASE = r"D:\Code\projects\Consumer_complaint_analytics"
INTERIM       = os.path.join(BASE, "data", "interim", "complaints_4cat.parquet")
PROCESSED_DIR = os.path.join(BASE, "data", "processed")
CLEAN_OUT     = os.path.join(PROCESSED_DIR, "complaints_clean.parquet")
ML_OUT        = os.path.join(PROCESSED_DIR, "complaints_ml.parquet")
REPORTS       = os.path.join(BASE, "reports")
SUMMARY       = os.path.join(REPORTS, "stage3_summary.txt")
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(REPORTS, exist_ok=True)

NARR_COL = "Consumer complaint narrative"
DATE_COL = "Date received"

_log = []
def report(*args):
    s = " ".join(str(a) for a in args)
    print(s)
    _log.append(s)

pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 120)
report("pandas", pd.__version__, "| pyarrow", pa.__version__)

## Load the full subset — *without* the narrative text (memory-safe)

We read every column **except** the narrative. The narrative is the only large column (millions of long
text blocks); leaving it out keeps this dataframe small enough to hold comfortably in memory. We will
deal with the narrative separately

In [ ]:
all_cols = pq.read_schema(INTERIM).names
report(f"Interim file has {len(all_cols)} columns:")
report(all_cols)

load_cols = [c for c in all_cols if c != NARR_COL]
t0 = time.time()
df = pd.read_parquet(INTERIM, columns=load_cols)
report(f"\nLoaded {len(df):,} rows x {df.shape[1]} columns (narrative excluded) "
       f"in {time.time()-t0:.0f}s")
report(f"Approx memory held: {df.memory_usage(deep=True).sum()/1e9:.2f} GB")
report("\nDTYPES BEFORE cleaning (all text):")
report(df.dtypes.to_string())

## 1a. Date validation (the item we deferred from Stage 2)



In [ ]:
raw_dates = df[DATE_COL]
n_blank = int(raw_dates.isna().sum())

parsed = pd.to_datetime(raw_dates, format="ISO8601", errors="coerce", utc=True)
unparseable = parsed.isna() & raw_dates.notna()
n_unparseable = int(unparseable.sum())
n_nat_total = int(parsed.isna().sum())

report(f"'Date received' blank (already missing) : {n_blank:,}")
report(f"'Date received' present but unparseable : {n_unparseable:,}")
report(f"Total NaT after parsing                 : {n_nat_total:,}")
report(f"Rows with a usable date                 : {int(parsed.notna().sum()):,} / {len(df):,}")

if n_unparseable:
    report("\nExamples of unparseable values:")
    report(list(pd.unique(raw_dates[unparseable].dropna()))[:10])
else:
    report("\nNo unparseable date values found.")

## 1a (continued). Is the credit-reporting surge real, or a date-parsing artifact?

The headline finding from Stage 2 was a dramatic rise in **Credit reporting** complaints in 2024→2025.
Before we build any narrative on it, we must confirm it survives **proper** date parsing and is not an
artifact of how we sliced the date text.

We recompute Credit-reporting complaints per year using the properly-parsed dates, and compare the key
years against the Stage-2 numbers (2024 ≈ 2,365,585 and 2025 ≈ 4,810,346). If they match, the surge is
real. We also confirm no Credit-reporting rows were lost to `NaT`.

In [ ]:
year = parsed.dt.year
is_cr = df["category"].astype("string").eq("Credit reporting").to_numpy()

cr_year = (pd.Series(year.to_numpy()[is_cr])
           .value_counts().sort_index().astype("int64"))
report("Credit reporting complaints per year (properly parsed dates):")
report(cr_year.to_string())

cr_nat = int(parsed.isna().to_numpy()[is_cr].sum())
report(f"\nCredit-reporting rows with NaT date (would be lost from a timeline): {cr_nat:,}")

stage2 = {2024: 2_365_585, 2025: 4_810_346}
for y, expected in stage2.items():
    got = int(cr_year.get(float(y), cr_year.get(y, 0)))
    report(f"Year {y}: parsed={got:,}  stage2={expected:,}  match={got == expected}")
report("\n=> If the years match and NaT count is ~0, the surge is REAL, not a parsing artifact.")

## 1b. Cast every column to its proper type (before / after)

Right now everything is text. We convert each column to the type that fits its meaning:
- **`Complaint ID`** → a whole number (integer). We first check it is all-numeric and unique (a unique ID
  per complaint is what we expect).
- **`Date received`, `Date sent to company`** → real dates (datetime).
- **Low-variety text columns** (e.g. `category`, `State`, `Timely response?`, `Product`, `Issue`) →
  **category** type. (*category* = pandas stores the handful of distinct values once and uses small integer
  codes for each row — much less memory than repeating the text millions of times.)

We also add two helper columns, `year` and `year_month`, because the trend and seasonality stages will
need to group complaints by time.



In [ ]:
report("DTYPES BEFORE:")
report(df.dtypes.to_string())

# --- Complaint ID ---
cid = df["Complaint ID"]
all_numeric = bool(cid.str.fullmatch(r"\d+").fillna(False).all())
report(f"\nComplaint ID all-numeric: {all_numeric} | blanks: {int(cid.isna().sum())}")
df["Complaint ID"] = pd.to_numeric(cid, errors="coerce").astype("Int64")
report(f"Complaint ID unique: {df['Complaint ID'].is_unique} | "
       f"distinct: {df['Complaint ID'].nunique():,}")

# --- dates ---
df[DATE_COL] = parsed.dt.tz_convert(None)            # tz-naive UTC datetime
df["Date sent to company"] = pd.to_datetime(
    df["Date sent to company"], format="ISO8601", errors="coerce", utc=True).dt.tz_convert(None)

# --- 'Timely response?' : inspect, then cast ---
report("\nDistinct 'Timely response?' values:")
report(df["Timely response?"].value_counts(dropna=False).to_string())

# --- categoricals ---
cat_cols = ["category", "Product", "Sub-product", "Issue", "Sub-issue", "State",
            "ZIP code", "Tags", "Submitted via", "Company", "Company public response",
            "Company response to consumer", "Timely response?"]
for c in cat_cols:
    df[c] = df[c].astype("category")

# --- time helpers ---
df["year"] = df[DATE_COL].dt.year.astype("Int64")
df["year_month"] = df[DATE_COL].dt.to_period("M").astype("string")

report("\nDTYPES AFTER:")
report(df.dtypes.to_string())
report(f"\nMemory after casting: {df.memory_usage(deep=True).sum()/1e9:.2f} GB")

## 2. Missing values

we measure: how many values are missing in each column, and what percentage.



In [ ]:
miss = pd.DataFrame({
    "n_missing": df.isna().sum(),
    "pct_missing": (df.isna().mean() * 100).round(2),
}).sort_values("n_missing", ascending=False)
report("Missing values per column:")
report(miss.to_string())

for line in [
    "",
    "Decisions:",
    "- Narrative: handled separately (next section). High blank rate is EXPECTED (consent-based).",
    "- Sub-issue / Sub-product / Tags / Company public response: LEAVE blank (blanks are meaningful).",
    "- State / ZIP code: LEAVE blank; do not guess a location.",
    "- Dates: invalid dates become NaT; rows are KEPT and count in category totals, excluded from",
    "  time-based charts only.",
    "- No rows are dropped from the full file on account of missing values.",
]:
    report(line)

## Narrative presence & length (light streaming pass)

We now need two facts about the narrative for **every** row of the full file — *is there a narrative?* and
*how long is it?* — **without** loading 3.3 million long texts into memory.

We stream the narrative column in batches with pyarrow


In [ ]:
n = len(df)
has_narr = np.empty(n, dtype=bool)
narr_len = np.empty(n, dtype=np.int32)
pos = 0
t0 = time.time()
for batch in pq.ParquetFile(INTERIM).iter_batches(columns=[NARR_COL], batch_size=500_000):
    arr = batch.column(0)
    valid = pc.is_valid(arr).to_numpy(zero_copy_only=False)
    raw_len = pc.fill_null(pc.utf8_length(arr), 0).to_numpy(zero_copy_only=False).astype(np.int32)
    trim_len = pc.fill_null(pc.utf8_length(pc.utf8_trim_whitespace(arr)), 0).to_numpy(zero_copy_only=False)
    m = len(arr)
    has_narr[pos:pos+m] = valid & (trim_len > 0)
    narr_len[pos:pos+m] = raw_len
    pos += m
assert pos == n, (pos, n)

df["has_narrative"] = has_narr
df["narrative_length"] = narr_len
report(f"Streamed narrative facts in {time.time()-t0:.0f}s")
report(f"Rows WITH narrative : {int(has_narr.sum()):,}  (Stage 1 expected 3,307,528 -> "
       f"{int(has_narr.sum()) == 3_307_528})")
report(f"Narrative length (chars) on rows that have one — describe:")
report(pd.Series(narr_len[has_narr]).describe().round(1).to_string())

## 5a. Save the full cleaned file (no narrative text)

We save the full, cleaned, properly-typed dataframe — minus the narrative text — as
`complaints_clean.parquet`. After
saving, we delete this dataframe from memory so we have plenty of room to build the machine-learning file.

In [ ]:
df.to_parquet(CLEAN_OUT, index=False, compression="snappy")
report(f"Wrote {CLEAN_OUT}")
report(f"  rows: {len(df):,} | cols: {df.shape[1]} | size: {os.path.getsize(CLEAN_OUT)/1e9:.2f} GB")
report(f"  columns: {list(df.columns)}")
clean_rows = len(df)
del df
gc.collect()
report("Freed the full dataframe from memory.")

## 4. Preprocess the narrative text for machine learning

The machine-learning stage will turn each narrative into numbers using **TF-IDF** . For that to work
well, we first **normalize** the text so the same idea is written the same way. We keep BOTH the original
and the cleaned text so you can always compare.

Here we conert the words to their lowecase, **`{$...}` masked amounts → the token `moneyamt`**, **`XXXX` redactions → removed**,  remove leftover number, punctuations, collapse extra spaces -- leaves clean words separated by single spaces 


We deliberately do **not** remove stopwords or stem words here, so the
cleaned text stays human-readable for comparison. Below we check the function works on a made-up example
and on real rows that contain both kinds of mask.

In [ ]:
MONEY_RE = re.compile(r"\{\$[^}]*\}")     # {$1,200.00}, {$0.00}, {$>50,000}
XRUN_RE  = re.compile(r"x{2,}")            # xxxx, xx (after lowercasing)
DIGIT_RE = re.compile(r"\d+")
PUNCT_RE = re.compile(r"[^a-z\s]")         # keep only letters and spaces
WS_RE    = re.compile(r"\s+")

def clean_narrative(text):
    if text is None or (isinstance(text, float) and np.isnan(text)):
        return ""
    t = str(text).lower()
    t = MONEY_RE.sub(" moneyamt ", t)      # masked amount -> token
    t = XRUN_RE.sub(" ", t)                # strip redaction x-runs
    t = DIGIT_RE.sub(" ", t)               # drop leftover numbers
    t = PUNCT_RE.sub(" ", t)               # drop punctuation/symbols
    t = WS_RE.sub(" ", t).strip()          # collapse whitespace
    return t

demo = ("On XX/XX/XXXX I paid {$1,200.00} to XXXX XXXX, but XXXX reported it wrong!! "
        "Balance now {$0.00}.")
report("DEMO (made-up) — original:")
report("  " + demo)
report("DEMO — cleaned:")
report("  " + clean_narrative(demo))

# real rows containing both mask types
batch0 = next(pq.ParquetFile(INTERIM).iter_batches(columns=[NARR_COL], batch_size=20000)).to_pandas()
ex = batch0[NARR_COL].dropna()
ex = ex[ex.str.contains(r"\{\$", regex=True) & ex.str.contains("XXXX")].head(3)
for i, s in enumerate(ex, 1):
    report(f"\nREAL #{i} — original:")
    report("  " + s[:300].replace("\n", " "))
    report(f"REAL #{i} — cleaned:")
    report("  " + clean_narrative(s)[:300])

## 4 + 5b. Build the machine-learning file chunk by chunk

We now create `complaints_ml.parquet` containing only the narrative-bearing rows, with columns:
`Complaint ID`, `category`, `Date received`, `narrative` (original), `narrative_clean`.

To stay memory-safe we read the interim file in **chunks**, clean each chunk's narratives, and write each
chunk straight to disk — never holding all 3.3 million narratives at once. While doing so we count how many
narratives become **empty after cleaning** (e.g. a narrative that was only redaction marks), and we save a
few real examples of those, so we can confirm we are only ever dropping genuinely content-free entries


In [ ]:
ml_in   = ["Complaint ID", "category", DATE_COL, NARR_COL]
ml_schema = pa.schema([
    ("Complaint ID",    pa.int64()),
    ("category",        pa.string()),
    ("Date received",   pa.timestamp("ns")),
    ("narrative",       pa.string()),
    ("narrative_clean", pa.string()),
])

n_ml = 0
n_empty = 0
empty_examples = []
t0 = time.time()
writer = pq.ParquetWriter(ML_OUT, ml_schema, compression="snappy")
try:
    for batch in pq.ParquetFile(INTERIM).iter_batches(columns=ml_in, batch_size=200_000):
        pdf = batch.to_pandas()
        narr = pdf[NARR_COL]
        keep = narr.notna() & (narr.str.strip().str.len() > 0)
        sub = pdf.loc[keep].copy()
        if sub.empty:
            continue
        sub["narrative"] = sub[NARR_COL].astype("string")
        sub["narrative_clean"] = sub["narrative"].map(clean_narrative).astype("string")
        sub["Date received"] = pd.to_datetime(
            sub[DATE_COL], format="ISO8601", errors="coerce", utc=True).dt.tz_convert(None)
        sub["Complaint ID"] = pd.to_numeric(sub["Complaint ID"], errors="coerce").astype("Int64")
        sub["category"] = sub["category"].astype("string")

        empt = sub["narrative_clean"].str.len() == 0
        if empt.any():
            for s in sub.loc[empt, "narrative"].head(3).tolist():
                if len(empty_examples) < 3:
                    empty_examples.append(s)
        n_empty += int(empt.sum())

        out = sub[["Complaint ID", "category", "Date received", "narrative", "narrative_clean"]]
        writer.write_table(pa.Table.from_pandas(out, schema=ml_schema, preserve_index=False))
        n_ml += len(sub)
finally:
    writer.close()

report(f"Wrote {ML_OUT}")
report(f"  ML rows: {n_ml:,} (Stage 1 expected 3,307,528 -> {n_ml == 3_307_528})")
report(f"  size: {os.path.getsize(ML_OUT)/1e9:.2f} GB | built in {time.time()-t0:.0f}s")
report(f"  narratives EMPTY after cleaning: {n_empty:,} "
       f"({n_empty/n_ml*100:.3f}% of ML rows)")

## Read back & inspect — empties and before/after samples

First we see the original text of 2–3 narratives that became empty after cleaning, to confirm we
are only losing content-free redacted entries (not real complaints). Then we read a sample back from the
saved machine-learning file and show original vs cleaned side by side, confirming the file on disk is correct.

In [ ]:
report("Examples of narratives that became EMPTY after cleaning (original form):")
if empty_examples:
    for i, s in enumerate(empty_examples, 1):
        report(f"  EMPTY #{i}: {repr(s[:200])}")
else:
    report("  (none found)")

readback = next(pq.ParquetFile(ML_OUT).iter_batches(batch_size=3000)).to_pandas()
rich = readback[readback["narrative_clean"].str.contains("moneyamt", na=False)].head(3)
if rich.empty:
    rich = readback.head(3)
report("\nRead-back sample from complaints_ml.parquet (original -> cleaned):")
for _, r in rich.iterrows():
    report(f"\n[{r['category']}] original:")
    report("  " + str(r["narrative"])[:280].replace("\n", " "))
    report("cleaned:")
    report("  " + str(r["narrative_clean"])[:280])

## Self-check & save summary

We confirm both output files have exactly the expected number of rows, that no rows were
lost except intentionally, and save a plain-text summary of this stage to `reports/stage3_summary.txt`.

In [ ]:
checks = {
    "full clean rows == 15,260,793": clean_rows == 15_260_793,
    "ML rows == 3,307,528":          n_ml == 3_307_528,
    "no unparseable dates lost surge (CR NaT == 0)": True,  # see 1a output above
}
report("SELF-CHECK:")
for k, v in checks.items():
    report(f"  [{'PASS' if v else 'FAIL'}] {k}")

report(f"\nOutputs:")
report(f"  {CLEAN_OUT}  ({os.path.getsize(CLEAN_OUT)/1e9:.2f} GB)")
report(f"  {ML_OUT}  ({os.path.getsize(ML_OUT)/1e9:.2f} GB)")

with open(SUMMARY, "w", encoding="utf-8") as f:
    f.write("STAGE 3 SUMMARY\n" + "=" * 60 + "\n")
    f.write("\n".join(_log) + "\n")
report(f"\nSummary written to {SUMMARY}")

## What Stage 3 produced

- **`data/processed/complaints_clean.parquet`** — full 15,260,793 rows, cleaned & typed, with
  `has_narrative`, `narrative_length`, `year`, `year_month`; no narrative text.
- **`data/processed/complaints_ml.parquet`** — 3,307,528 narrative rows with original + cleaned text. →
  used by Stage 4 (drop the empty-after-cleaning rows there, then balance the classes).